<a href="https://colab.research.google.com/github/lucasalmo/lia1_2026_1_new/blob/main/Aula%2010%20-%20Modelo%20para%20Identificar%20Cats%20or%20Dogs/LIA_Cats%26Dogs_com_PyTorch_ViT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🐶🐱 Classificador de Cães e Gatos com Inteligência Artificial

Bem-vindo a este projeto! Aqui vamos construir um sistema de **Inteligência Artificial capaz de olhar para uma foto e dizer se ela é de um cachorro ou de um gato** — com alta precisão.

Mas como isso é possível? A resposta está em uma técnica chamada **aprendizado profundo** (*deep learning*), que permite ensinar um computador a reconhecer padrões em imagens da mesma forma que nós, humanos, aprendemos a distinguir objetos ao longo da vida.

---

## 🧠 A grande sacada: não precisamos começar do zero!

Treinar uma IA do zero para reconhecer imagens exigiria **milhões de fotos e semanas de processamento**. Por isso, usamos uma estratégia muito mais inteligente chamada **Transfer Learning** (Transferência de Aprendizado).

Imagine que você já sabe dirigir um carro. Aprender a dirigir uma moto é muito mais fácil do que para alguém que nunca dirigiu nada, certo? A mesma lógica se aplica aqui: usamos um modelo de IA que já "aprendeu a enxergar" analisando **mais de 14 milhões de imagens** (o famoso dataset ImageNet), e apenas ensinamos ele a distinguir cães de gatos. Esse processo de adaptação é chamado de **fine-tuning**.

---

## 🔬 Qual modelo estamos usando?

Usamos o **Vision Transformer (ViT)**, criado pelo Google. Ele representa uma revolução na visão computacional: em vez de analisar a imagem pixel por pixel, o ViT a divide em pequenos **blocos (patches)** de 16×16 pixels e analisa como esses blocos se relacionam entre si — como se estivesse lendo as "palavras" de uma imagem.

---

## 📦 Os dados que vamos usar

| Conjunto | Quantidade | Composição |
|----------|-----------|------------|
| Treino | 1.002 imagens | 501 cães + 501 gatos |
| Teste | 200 imagens | 100 cães + 100 gatos |

> ⚠️ **Importante:** os datasets serão baixados automaticamente do repositório da disciplina nas células abaixo.

## ⚙️ Etapa 1 — Preparando o Ambiente

Antes de começar a trabalhar com IA, precisamos instalar algumas **ferramentas especializadas**. Pense nisso como instalar programas no seu computador antes de usar.

Vamos instalar três bibliotecas principais:

| Biblioteca | Para que serve |
|-----------|----------------|
| `transformers` | Dá acesso a modelos de IA prontos, como o ViT do Google |
| `datasets` | Ajuda a carregar e organizar conjuntos de dados |
| `evaluate` | Fornece ferramentas para medir o desempenho do modelo |

> 💡 O `-q` no comando faz a instalação em modo silencioso, sem exibir mensagens desnecessárias.

In [ ]:
# Instalar pacotes necessários
!pip install -q transformers datasets evaluate

## 📚 Etapa 3 — Importando as Ferramentas

Agora vamos carregar todas as bibliotecas que usaremos ao longo do projeto. É como separar todas as ferramentas antes de começar um trabalho.

| Ferramenta | O que ela faz |
|-----------|---------------|
| `torch` (PyTorch) | O "motor" principal do nosso projeto — executa todos os cálculos da IA |
| `DataLoader` | Organiza as imagens em grupos (batches) para alimentar o modelo aos poucos |
| `ImageFolder` | Lê automaticamente as imagens das pastas e identifica as classes pelo nome das pastas |
| `ViTForImageClassification` | O modelo ViT já configurado para classificar imagens |
| `ViTImageProcessor` | Prepara as imagens no tamanho e formato exatos que o ViT precisa receber |
| `tqdm` | Exibe uma barra de progresso durante o treinamento |
| `sklearn` | Usada para gerar a matriz de confusão ao final |

Também verificamos se há uma **GPU disponível**. A GPU (placa de vídeo) é muito mais rápida que a CPU para treinar modelos de IA — fazendo a diferença entre treinar em minutos vs. horas.

> 💡 No Google Colab, ative a GPU em: **Ambiente de execução → Alterar tipo de ambiente de execução → T4 GPU**

In [ ]:
# Importação de bibliotecas
import torch
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision.datasets import ImageFolder
from transformers import ViTForImageClassification, ViTImageProcessor
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
# Verificar se GPU está disponível
# Caso esteja usando CPU, o treinamento será mais lento
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Usando dispositivo: {device}')

## 🤖 Etapa 4 — Carregando o Modelo de IA

Chegou a hora de carregar o "cérebro" do nosso projeto: o **Vision Transformer (ViT)**.

### O que é o Vision Transformer?

O ViT foi criado pelo Google em 2020 e trouxe uma ideia revolucionária para a visão computacional. Veja como ele funciona:

**1. Divisão em patches:**
A imagem é dividida em pequenos blocos de **16×16 pixels**, chamados de *patches*. Uma imagem de 224×224 pixels gera 196 patches.

**2. Leitura como uma sequência:**
Esses patches são processados em sequência, como se fossem as palavras de uma frase. Isso permite que o modelo entenda relações entre partes distantes da imagem — por exemplo, associar as orelhas ao focinho de um cachorro.

**3. Atenção global:**
Usando o mecanismo de **atenção** (o mesmo usado em modelos de linguagem como o GPT), o ViT aprende quais partes da imagem são mais importantes para tomar a decisão.

### Por que reutilizar um modelo pronto?

O modelo `google/vit-base-patch16-224` já foi treinado com **14 milhões de imagens** do ImageNet. Isso significa que ele já sabe reconhecer bordas, texturas, formas, olhos, patas, etc. Tudo que precisamos fazer é **trocar a última camada** — que originalmente classificava 1.000 categorias — por uma nova com apenas **2 saídas: gato ou cachorro**.

> 🔧 O parâmetro `ignore_mismatched_sizes=True` permite essa substituição sem causar erros.

In [ ]:
# Nome do modelo pré-treinado
model_name = "google/vit-base-patch16-224"

# Carregar processador e modelo
processor = ViTImageProcessor.from_pretrained(model_name)
model = ViTForImageClassification.from_pretrained(
    model_name,
    num_labels=2,                 # 2 classes: gato ou cachorro
    ignore_mismatched_sizes=True  # permite ajustar camadas finais
)
model = model.to(device)
print('✅ Modelo carregado com sucesso!')

## 🗂️ Etapa 5 — Preparando os Dados para o Treinamento

Antes de treinar, precisamos organizar as imagens de um jeito que o modelo consiga consumir eficientemente. Fazemos isso em três passos:

### Passo 1 — Dataset personalizado
Criamos uma classe chamada `DogsAndCatsDataset` que funciona como um "leitor de imagens inteligente". Para cada foto, ela:
- Abre a imagem
- Aplica o **pré-processamento do ViT**: redimensiona para 224×224 pixels e normaliza os valores de cor
- Retorna a imagem pronta junto com seu rótulo (0 = gato, 1 = cachorro)

### Passo 2 — Divisão treino / validação
Das 1.002 imagens de treino, separamos:

| Parte | Proporção | Quantidade | Uso |
|-------|-----------|-----------|-----|
| Treino | 80% | ~801 imagens | O modelo **aprende** com essas imagens |
| Validação | 20% | ~201 imagens | Verificamos se o modelo está **generalizando** bem |

Essa divisão é fundamental: sem ela, não saberíamos se o modelo apenas decorou as fotos (*overfitting*) ou se realmente aprendeu a distinguir cães de gatos.

### Passo 3 — DataLoader
O `DataLoader` organiza as imagens em **lotes de 32** (batches). Em vez de processar 801 imagens de uma vez — o que sobrecarregaria a memória —, o modelo processa 32 por vez, de forma eficiente na GPU.

In [ ]:
# Criar o dataset personalizado
class DogsAndCatsDataset(Dataset):
    def __init__(self, root_dir, processor):
        self.dataset = ImageFolder(root_dir)
        self.processor = processor

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        # Pré-processar imagem para o ViT
        inputs = self.processor(images=image, return_tensors="pt")
        pixel_values = inputs['pixel_values'].squeeze()
        return pixel_values, label

In [ ]:
# Diretório de treino
train_dir = './dataset_treino'

# Criar dataset completo
full_dataset = DogsAndCatsDataset(train_dir, processor)

# Dividir em treino (80%) e validação (20%)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

print(f'Total de imagens: {len(full_dataset)}')
print(f'Treinamento: {len(train_dataset)} | Validação: {len(val_dataset)}')

# DataLoader
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

## 🏋️ Etapa 6 — Treinamento do Modelo

Esta é a etapa central do projeto! Aqui o modelo vai efetivamente **aprender** a distinguir cães de gatos.

### Como o modelo aprende?

O processo de aprendizado funciona como um ciclo de tentativa e erro:

```
Ver imagens → Fazer previsão → Calcular o erro → Corrigir os pesos → Repetir
```

**1. Ver imagens:** o modelo recebe um lote de 32 fotos

**2. Fazer previsão:** ele tenta adivinhar se cada foto é de um gato ou cachorro

**3. Calcular o erro (loss):** comparamos a previsão com a resposta correta. Quanto maior o erro, mais o modelo precisa se ajustar

**4. Corrigir os pesos (backpropagation):** o otimizador **AdamW** ajusta os milhões de parâmetros internos do modelo para reduzir o erro na próxima tentativa

**5. Repetir:** esse ciclo se repete para todas as imagens. Uma passagem completa pelo dataset é chamada de **época (epoch)**

### Configurações do treinamento

| Parâmetro | Valor | Significado |
|-----------|-------|-------------|
| `num_epochs` | 10 | O modelo vai estudar todas as imagens 10 vezes |
| `learning_rate` | 5e-5 | Quão grandes são os ajustes a cada passo (valor pequeno = aprendizado mais estável) |
| `batch_size` | 32 | Quantas imagens são processadas por vez |

> ⏱️ Com a GPU T4 do Google Colab, cada época leva cerca de **1 a 2 minutos**. O treinamento completo (10 épocas) deve levar entre 10 e 20 minutos.

In [ ]:
# Configurações de treino
num_epochs = 10
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

# Loop de treinamento
for epoch in range(num_epochs):
    model.train()
    total_loss, correct, total = 0, 0, 0

    progress_bar = tqdm(train_loader, desc=f'Epoch {epoch + 1}/{num_epochs}')
    for images, labels in progress_bar:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(pixel_values=images, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = outputs.logits.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        progress_bar.set_postfix(
            loss=total_loss/(len(progress_bar)),
            accuracy=100.*correct/total
        )

    # Validação
    model.eval()
    val_correct, val_total, val_loss = 0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(pixel_values=images, labels=labels)
            val_loss += outputs.loss.item()
            _, predicted = outputs.logits.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    print(f"📊 Epoch {epoch+1}/{num_epochs}")
    print(f"  Treino: Loss {total_loss/len(train_loader):.4f} | Acc {100.*correct/total:.2f}%")
    print(f"  Val   : Loss {val_loss/len(val_loader):.4f} | Acc {100.*val_correct/val_total:.2f}%\n")

## 📊 Etapa 7 — Avaliando o Desempenho: Matriz de Confusão

Depois de treinar, precisamos saber: **o modelo realmente aprendeu?** A matriz de confusão é uma das melhores ferramentas para isso.

### O que é uma matriz de confusão?

É uma tabela que mostra **quantas vezes o modelo acertou e errou** para cada classe:

| | 🐱 Previu: Gato | 🐶 Previu: Cachorro |
|---|---|---|
| **🐱 Era: Gato** | ✅ Acerto (Verdadeiro Positivo) | ❌ Erro (Falso Negativo) |
| **🐶 Era: Cachorro** | ❌ Erro (Falso Positivo) | ✅ Acerto (Verdadeiro Positivo) |

### Como interpretar?

- Os números na **diagonal** (canto superior esquerdo e inferior direito) representam os **acertos**
- Os números **fora da diagonal** representam os **erros**
- Quanto mais os valores estiverem concentrados na diagonal, **melhor o modelo!**

Por exemplo: se na linha "Era: Gato" o modelo acertou 95 e errou 5, significa que ele identificou corretamente 95% dos gatos.

In [ ]:
# Gerar a Matriz de Confusão
all_preds, all_labels = [], []
model.eval()
with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(pixel_values=images, labels=labels)
        _, predicted = outputs.logits.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Matriz de confusão (0 = cat; 1 = dog)
cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['cat','dog'])
disp.plot(cmap='Blues')
plt.title("Matriz de Confusão")
plt.show()

## 🚀 Etapa 8 — Testando com Novas Imagens (Deploy)

Com o modelo treinado, chegou a hora mais divertida: **testar com fotos reais!**

Aqui vamos carregar uma imagem do dataset de teste e ver o que a IA diz. Isso é chamado de **inferência** — usar um modelo já treinado para fazer previsões em dados novos que ele nunca viu antes.

### O que acontece por baixo dos panos?

Quando o modelo recebe uma foto, o seguinte ocorre em milissegundos:

| Etapa | O que acontece |
|-------|----------------|
| 1. Pré-processamento | A imagem é redimensionada para 224×224 e normalizada |
| 2. Forward pass | A imagem percorre todas as camadas do ViT |
| 3. Logits | O modelo gera dois números — um score para gato e um para cachorro |
| 4. Softmax | Os scores são convertidos em probabilidades que somam 100% |
| 5. Decisão | A classe com maior probabilidade é a resposta final |

### Exemplo de saída esperada:
```
🐾 Classe predita: dog | Confiança: 98.7%
```

> 💡 **Confiança alta** (acima de 90%) indica que o modelo tem certeza da previsão. **Confiança baixa** (abaixo de 60%) pode indicar uma imagem ambígua ou de baixa qualidade.

In [ ]:
from PIL import Image

# Carregar imagem de teste
test_image_path = './dataset_teste/100.jpg'
image = Image.open(test_image_path)

# Pré-processar
inputs = processor(images=image, return_tensors="pt").to(device)

# Inferência
model.eval()
with torch.no_grad():
    outputs = model(**inputs)
    probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)
    predicted_class = torch.argmax(probabilities, dim=-1).item()

class_names = ['cat', 'dog']
predicted_label = class_names[predicted_class]
confidence = probabilities[0][predicted_class].item()

print(f"🐾 Classe predita: {predicted_label} | Confiança: {confidence:.2%}")
image